# Calculating excited-state properties with **`exciting`** using one-shot GW
By: Martí Raya Moreno

**<span style="color:firebrick">Note</span>**: Due to time constraints, all results used in this tutorial have been precomputed. Therefore, the corresponding data are retrieved from the [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries) database. Otherwise, `exciting` would need to be installed and compiled, and the appropriate environment would need to be configured. More information about exciting and its usage can be found on [exciting webpage](https://www.exciting-code.org/).

<hr style="border:2px solid #DDD"> </hr>

# Introduction to the $GW$ Method

The $GW$ method is a **many-body perturbation theory (MBPT) approach** that goes beyond standard density functional theory (DFT) for describing electronic excitations. While Kohn-Sham DFT is, in principle, exact for ground-state densities and energies, it is **not designed to accurately predict excited-state properties**. In particular, Kohn-Sham eigenvalues, introduced as Lagrange multipliers, cannot be rigorously interpreted as quasiparticle (QP) energies. They often provide a rough estimate, but can differ substantially from experimental spectra measured in techniques like X-ray photoemission spectroscopy (XPS) or angle-resolved photoemission spectroscopy (ARPES).

## What the $GW$ Method Does

The $GW$ method **simulates charged excitations** by incorporating nonlocal and dynamical screening effects caused by adding or removing an electron. In practical terms, it **computes quasiparticle energies** that are closer to what is observed experimentally, unlike a single DFT calculation with fixed particle number. The most common variant, the **single-shot $G_0W_0$**, calculates the QP energies perturbatively starting from a mean-field reference (DFT, HF, or hybrid functional), approximating the many-body self-energy $\Sigma$ as:

$$
\Sigma \approx i G W
$$

where $G$ is the single-particle Green’s function, $W$ is the screened Coulomb interaction, and the vertex function is set to unity ($\Gamma \approx 1$). This approximation reduces the full complexity of Hedin’s equations to a tractable problem.

## How the $GW$ Method Works

The practical workflow of $G_0W_0$ involves several steps:

1. **Reference Calculation:** Start from a mean-field solution (DFT/HF/hybrid) to obtain Kohn-Sham eigenvalues and wavefunctions.  
2. **Polarizability and Dielectric Function:** Construct the polarizability using products of single-particle states, often expanded in a mixed-product basis. Compute the dielectric function within the Random Phase Approximation (RPA).  
3. **Screened Coulomb Interaction:** Obtain the screened Coulomb interaction $W$ from the inverse dielectric function.  
4. **Self-Energy Evaluation:** Compute the correlation and exchange parts of the self-energy. The total self-energy is then used in the **linearized quasiparticle equation**:

$$
\epsilon^{\rm QP}_{n\mathbf{k}} \approx \epsilon_{n\mathbf{k}}^{\rm KS} + Z_{n\mathbf{k}} \langle \psi_{n\mathbf{k}} | \Sigma(\epsilon_{n\mathbf{k}}^{\rm KS}) - v_{\rm XC} | \psi_{n\mathbf{k}} \rangle
$$

where $Z_{n\mathbf{k}}$ is a renormalization factor accounting for the QP weight. 

## What Output to Expect

The main output of a $GW$ calculation is the **quasiparticle energy spectrum**, which can be directly compared to experimental measurements of electron addition and removal. Compared to Kohn-Sham DFT, $GW$ energies typically:

- Correct the underestimation of band gaps in semiconductors and insulators  
- Provide more accurate ionization potentials 

In summary, the $GW$ method bridges the gap between ground-state DFT and experimental spectroscopies by simulating the **dynamical response of electrons**, while relying on controlled approximations (single-shot perturbation, RPA, vertex neglect) to make the computation feasible.

# Task-Based Workflow for $GW$ Calculations

![GW workflow](data/GW_tasks.png)

The $GW$ method is implemented as a **task-based workflow**, where each computational step is treated as an independent task. This modular approach allows **parallelization over multiple dimensions**—$\mathbf{k}$ points, $\mathbf{q}$ points, and bands—and supports **GPU acceleration** for the most compute-intensive operations. By organizing $GW$ as a sequence of tasks, the workflow achieves strong and weak scaling across modern HPC architectures, enabling large-scale, high-precision calculations. We note that the preferred workflow is indicated by solid lines and exploits crystal symmetry to reduce the computational cost by limiting the calculation of the dielectric matrix to the irreducible wedge.

## Overview of the tasks:

1. **Coulomb**  
   - Evaluates the **bare Coulomb interaction** between electrons.  
   - Parallelized over **$\mathbf{q}$ points**.  
   - Supports GPU acceleration for the diagonalization of the bare Coulomb matrix.

2. **vxc**  
   - Computes the **diagonal matrix elements of the exchange-correlation potential** from the reference DFT calculation.  
   - Parallelized over **$\mathbf{k}$ points**.  
   - Lightweight task, usually CPU-bound.

3. **Polarizability**  
   - Constructs the **irreducible polarizability** by summing products of occupied and unoccupied states.  
   - Parallelized over **$\mathbf{q}$ points**.  
   - GPU acceleration is applied to dense matrix multiplications and loop-intensive operations.

4. **Epsilon**  
   - Forms the **dielectric matrix** in the Random Phase Approximation (RPA) using the polarizability.  
   - Parallelized over **$\mathbf{k}$ points, $\mathbf{q}$ points, unoccopied bands**.  
   - Matrix inversion steps can leverage GPU libraries.

5. **invertEpsilon**  
   - Computes the **inverse of the dielectric matrix**, required to screen the Coulomb interaction.  
   - Parallelized over **$\mathbf{q}$ points**.  

6. **irreducibleMapping**  
   - Exploits **crystal symmetries** to reconstruct the full inverse dielectric matrix in the Brillouin zone from the irreducible set.  
   - Parallelized over **$\mathbf{q}$ points**.  

7. **sigmax (Exchange Self-Energy)**  
   - Computes the **exchange part of the self-energy**, involving sums over occupied states.  
   - Parallelized over **$\mathbf{k}$ and $\mathbf{q}$-points**.  
   - GPU acceleration is applied to dense matrix multiplications and loop-intensive operations.

8. **sigmac (Correlation Self-Energy)**  
   - Computes the **correlation part of the self-energy**, involving sums over unoccupied states and frequency integration.  
   - Parallelized over **$\mathbf{k}$ points, $\mathbf{q}$ points, unoccopied bands**.  
   - GPU acceleration is applied to dense matrix multiplications and loop-intensive operations.

9. **QPEigenvalues**  
   - Combines the exchange and correlation contributions to compute **quasiparticle (QP) energies** using the linearized QP equation.  
   - Lightweight: no parallelized, can be run in serial.

# Importance of the Basis Functions and High-Energy Local Orbitals (LOs)

In all-electron methods such as **(L)APW+lo**, the **quality and completeness of the basis set** are absolutely critical—especially when considering that **excited-state properties depend much more strongly on the basis than ground-state ones**.

Ground-state (GS) calculations mainly require an accurate description of *occupied* and low-lying *unoccupied* states.  
However, excited-state calculations (optics, GW, BSE, RPA, dielectric functions, etc.) explicitly involve **empty bands**, often extending **tens of eV above the Fermi level**. These states are poorly represented even by a basis that is over-converged for the GS, to the point that the results become unreliable or even qualitatively wrong.

➡️ **Solution:** add **extra Local Orbitals (LOs)** at **high energies** to systematically improve basis flexibility and describe states over the relevant energy range.

## Preliminary GS calculations

As a preliminary step to calculate properties from GW, a ground-state calculation has to be performed. In this tutorial we consider as an example **$\beta$-Ga<sub>2</sub>O<sub>3</sub>**.

An `exciting` input file needs to contain the <code><span style="color:green">structure</span></code> element in which we include the lattice parameter, basis vectors, and atomic positions and the <code><span style="color:green">groundstate</span></code> element where we include parameters responsible for **DFT** ground-state calculations. We also need so called species files in which we define the augmented plane-wave and local-orbital basis. Have in mind that the basis within species files needs to be converged.

We can read the ground-state input from file using the exciting python interface **excitingtools**. Details on this can be found in the [**excitingtools tutorial**](https://www.exciting-code.org/uploads/exciting/tutorial_notebooks/tutorial_excitingtools.html).

In [ ]:
from excitingtools import ExcitingInputXML, ExcitingGWInput, ExcitingPropertiesInput
from excitingtools.input.input_classes import ExcitingBandStructureInput

exciting_input = ExcitingInputXML.from_xml('data/gs_input.xml')

How such an ground-state calculation can be run, you can find out in the ground-state tutorial. 
The main output of the **DFT** calculation for subsequent excited states calculations are stored in **STATE.OUT** and **EFERMI.OUT**. 

Since we use the mock runner and the binary file **STATE.OUT** is not transferable between maschines and compilers, we dont fetch **DFT** results seperately and directly proceed with the **GW** calculations. But we assume, we have the neccesary ground state calculation done, therefor we can skip the recomputation of the ground state by seting the following parameter in the input file. 

In [ ]:
exciting_input.groundstate.do = "skip"

# Running a GW Calculation

## Input definition

We create an input file to perfom an excited-state **GW** calculation. We define the parameters used in all of the tasks of the GW workflow. Like, for example, the size of the **q**-grid, the number of unoccupied bands, the mixed product basis, and many more. 

In [ ]:
mixbasis = {'lmaxmb': 4,
            'epsmb': 1.0e-4,
            'gmb': 1.0}

barecoul = {'pwm': 2.0,
            'barcevtol': 0.1}

gw = {'taskname': 'taskgroup',
      'ngridq': [4, 4, 4],
      'nempty':1600,
      'ibgw':1,
      'nbgw':120,
      'mblksiz':400,
      'GBatchCount':25,
      'skipgnd':False,
      'qdepw':'sum',
      'coreflag':'all',
      'mixbasis': mixbasis,
      'barecoul': barecoul,
      'scrcoul': {'averaging': 'anisotropic'},
      'freqgrid': {'nomeg': 32}}

exciting_input.gw = ExcitingGWInput(**gw)

## Dry run
A GW calculation in `exciting` should always begin with a **dry run** in order for it to run most efficient. This step determines the number of irreducible q‑points, checks the symmetry settings, and prints memory and parallelization information. It does **not** compute the GW quantities yet, it only prepares the workflow.
The details of paralelization are out of the scope of this tutorial. However, we collected some information on it in the toggled cells below. 

<details>
<summary><strong><span style="color:firebrick">$\Rightarrow$ Read more</span></strong></summary>
    
The GW dry run first performs a **non‑self-consistent (NSCF) calculation**, which generates the **Kohn–Sham (KS) eigenvalues and eigenvectors** on the selected k-point mesh. The calculation is carried out for the band range specified by the parameters `nempty` (number of empty/unoccupied states) and `ngridq` (q-point grid for $GW$). 

### Parallelization Table
It then prints a mapping of $q$‑points and how the workload is distributed across MPI ranks. Note that the distribution over MPI groups is controlled by `MPIDomainsQpoints` and `MPIDomainsKpoints` varibles within the given task.  

Example:

```
*** Table with index map
(xq)   q
  1    1
  2    2
  ...
 24   24
```

Then the dry‑run prints how the GW workload is distributed across MPI ranks.  
The table below shows the actual distribution for this system:

```
*** Parallelization
*** Overview of how indexes are distributed among MPI processes
*** (xq)i, (xq)f : first, last q-points indexes (as used in the code)
*** ki, kf       : first, last k-points indexes (as used in the code)
*** mi, mf       : first, last unoccupied states

 global   (xq)i   (xq)f      ki      kf      mi      mf
    ---       1      24       1      64      61    1660
```

Each MPI rank receives:

- **one q‑point** (because `(xq)i = (xq)f`)
- **a block of 8 k‑points**
- **the full empty‑state range** (`mi = 61`, `mf = 1660`)

This produces the following mapping:

```
rank   (xq)i (xq)f   ki   kf    mi    mf
  0       1     1     1    8    61  1660
  1       1     1     9   16    61  1660
  2       1     1    17   24    61  1660
  3       1     1    25   32    61  1660
  4       1     1    33   40    61  1660
  5       1     1    41   48    61  1660
  6       1     1    49   56    61  1660
  7       1     1    57   64    61  1660

  8       2     2     1    8    61  1660
  9       2     2     9   16    61  1660
 10       2     2    17   24    61  1660
 ...
191      24    24    57   64    61  1660
```
This distribution can be interpreted as:
- **q‑points are the outermost parallelization level**  
  Each q‑point is assigned to a group of 8 ranks (one group per q‑point).

- **Within each q‑point group, k‑points are distributed**  
  Each rank handles 8 k‑points.

- **Empty states are *not* distributed in this case** 
  Every rank processes the full unoccupied‑state range (`61–1660`). However, if more MPI processes are provided the code will further divide this into different ranks within the **k**-point group.

This table determines:

- how the ranks are distributed for the epsilon and sigma tasks  
- how to choose the number of MPI ranks per q‑point group

Interpretation:

- All ranks share the same q‑point range (1–24).
- k‑points are also shared (1–64).
- Empty states (`mi`–`mf`) are distributed across ranks.

Tasks with lower parallelization levels must be adjusted to match this distribution. In particular 
    
### Degenerate Subspace Truncation

In practical GW calculations, it is common to **truncate the number of bands** for computational efficiency and convergence. However, truncation must be done carefully because:

- There is a **risk of cutting a degenerate subspace**.
- If a degenerate subspace is truncated, the **dielectric matrix** and the **self-energy** may no longer respect the full crystal symmetry, potentially lifting degeneracies incorrectly.

---

### Practical Solution

- Adjust the number of empty bands **downward** to ensure that **complete degenerate subspaces** are included.  
- The `exciting` code handles this automatically and reports adjustments in the `GW_INFO.OUT` file. Example:

```
--------------------------------------------------------------------------------
-            Checking for possible degenerate subspace truncation              -
--------------------------------------------------------------------------------
 
    -Initial band truncation was         1661
    -Final band truncation has been adjusted to prevent the cutting of the degenerate subspaces to         1660

```

This indicates that **not all of the originally requested `nempty` bands are valid**. 

> ⚠️ Currently, the code does not automatically suggest a fully valid `nempty`. The user must ensure that the final effective `nempty` is sufficiently large. If in doubt, using the **full untruncated space** is recommended to avoid symmetry-breaking issues.


### Memory Estimates (`GW_MEMORY.yaml`)

Finally, the dry run generates a file (`GW_MEMORY.yaml`) containing the memory consumption; for instance:

```
task = epsilon, Memory estimates (in MB): 
  sgi:
    type: complex(dp)
    dims: [1848,1848]
    memory: 52
  mpwipw:
    type: complex(dp)
    dims: [1848,14682]
    memory: 414
  vmat:
    type: complex(dp)
    dims: [13990,13990]
    memory: 2986
  epsilon:
    type: complex(dp)
    dims: [13990,13990,32]
    memory: 95566
  fnm:
    type: complex(dp)
    dims: [98,1600,32,64]
    memory: 4900
  minmmat:
    type: complex(dp)
    dims: [13990,98,400]
    memory: 8368
  minm:
    type: complex(dp)
    dims: [13990,98,400]
    memory: 8368
  temp_calcminm2:
    type: complex(dp)
    dims: [1848,1848,25]
    memory: 1302
```

This indicates:
  - how much memory each rank needs  
  - how to adjust `mblksiz` and `GBatchCount` to fit memory limits  

Notes:

- Memory can be reduced by lowering:
  - `mblksiz="400"`
  - `GBatchCount="25"`
- Reducing these increases overhead but decreases peak memory.

---

### Summary

- The dry run is advised before any GW calculation.
- It determines:
  - irreducible q‑points  
  - symmetry usage
  - band truncation adjustment
  - parallel distribution with the current setup (i.e. `MPIDomainsQpoints` and `MPIDomainsKpoints`) 
  - memory requirements  
- Adjust `mblksiz` and `GBatchCount` to fit memory limits.




# Running a GW Calculation:

Because the different GW steps parallelize in different ways, it is recommended to run each task as an independent job. The typical sequence is:
  - I. Compute the bare Coulomb potential and the exchange-correlation potential
  - II. Compute the exchange part of the self-energy (sigmax)  
  - III. Compute the dielectric matrix  
  - IV. Invert the dielectric matrix and expand it to the full Brillouin zone  
  - V. Use these ingredients to compute the correlation part of the self-energy (sigmac)  
  - VI. Combine all contributions to obtain the quasiparticle (QP) energies  
  
Due to time constraints during this tutorial, we will show how to run all of the task at once. 
For this we have to add all the individual tasks to the task group in the input file.  


In [ ]:
qpoints = [{"qi": 1, "qf": -1}]
kpoints = [{"ki": 1, "kf": -1}]

coulomb_task = {'qpoints': qpoints}

vxc_task = {'qpoints': qpoints}

sigmax_task = {'kpoints': kpoints}

epsilon_task = {'usingIrreducibleWedge': True,
                'qpoints': qpoints}

invert_epsilon_task = {'qpoints': qpoints}

irreducible_mapping_task = {'qpoints': qpoints}

sigmac_task = {'kpoints': kpoints}

QPeigenvalues_task = {'FermiLevel': 'from_QP_Eigenvalues',
                      'kpoints': kpoints}

exciting_input.gw.taskGroup = {'outputFormat':'binary',
                               'dryRun':False,
                               'Coulomb': coulomb_task,
                               'sigmax': sigmax_task,
                               'epsilon': epsilon_task,
                               'invertEpsilon': invert_epsilon_task,
                               'irreducibleMapping': irreducible_mapping_task,
                               'sigmac': sigmac_task,
                               'QPEigenvalues': QPeigenvalues_task}

                              

For most of the taks we just specify the range of computed $q$- or $k$-points. In our case we want to compute all the $q$- and $k$-points for all tasks. For the computation of the dielectric matrix (epsilon) we set `'usingIrreducibleWedge': True` which activates symmetry reduction, so that only the irreducible wedge of the Brillouin zone is considered. This can significantly reduce computational cost. 
Then, we set `'outputFormat':'binary'` which means that output which is written out and read in between the individual tasks is written in binary, not text format. Also, we set `'dryRun':False`, since we want to do an actual calculation. A more in-depth description of all the possible input parameter can be found in the [**input reference**](http://exciting.wikidot.com/ref:input). 

Now, we generate a directory for the calculation to be run in

In [ ]:
from pathlib import Path
gw_dir = Path("gw_tutorial")
gw_dir.mkdir(exist_ok=True)

and save the generated input for exciting in an `input.xml` file. 

In [ ]:
input_xml = gw_dir / "input.xml"
exciting_input.write(input_xml)

The generated input file looks like this

In [ ]:
print(input_xml.read_text())

With that, we can run exciting using **excitingscripts**. This python library contains a lot of helpful scripts for running exciting and postprocessing exciting outputs. In this tutorial, we use the mock runner to fetch the pre-computed data from [**NOMAD**](https://nomad-lab.eu/prod/v1/gui/search/entries).

In [ ]:
%%bash
cd gw_tutorial
python -m excitingscripts.dpg2026.mock_execute_single GW --overwrite
cd ..


This will generate the `EVALQP.DAT` and its binary counterpart `EVALQP.OUT`. These files contain the quasiparticle energies on the irreducible k-point mesh. In order to generate the band structure, we need to modify our input file again, by adding the following to it.

In [ ]:
points = [{'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
         {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
         {'coord': [0.60315808, 0.60315808, 0.41000209], 'label':'F'},
         {'coord': [0.50000000, 0.50000000, 0.50000000], 'label':'L'},
         {'coord': [0.74187655, 0.25812345, 0.50000000], 'label':'I', 'breakafter':True},
         {'coord': [0.25812345, -0.25812345, 0.50000000], 'label':'I1'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'Z'},
         {'coord': [0.39684192, 0.39684192, 0.58999791], 'label':'F1', 'breakafter':True},
         {'coord': [0.50000000, 0.50000000, 0.00000000], 'label':'Y'},
         {'coord': [0.73364820, 0.26635180, 0.00000000], 'label':'X1', 'breakafter':True},
         {'coord': [0.26635180, -0.26635180, 0.00000000], 'label':'X'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'},
         {'coord': [0.50000000, 0.00000000, 0.00000000], 'label':'N', 'breakafter':True},
         {'coord': [0.50000000, 0.00000000, 0.50000000], 'label':'M'},
         {'coord': [0.00000000, 0.00000000, 0.00000000], 'label':'GAMMA'}]

In [ ]:
band_structure = ExcitingBandStructureInput(plot1d={'path':{'steps': 100, 'point': points}})

In [ ]:
exciting_input.properties = ExcitingPropertiesInput(bandstructure=band_structure)

In [ ]:
exciting_input.write(input_xml)

In [ ]:
print(input_xml.read_text())

**<span style="color:firebrick">Note</span>**: To avoid overhead, we already fetched the band-structure and DOS ouput files before. So we don't run exciting here. 

The generated band structure can be ploted using **excitingscripts**.

In [ ]:
%%bash
cd gw_tutorial
python -m excitingscripts.plot.band_structure
mv PLOT.png BS.png
cd ..

In [ ]:
from IPython.display import Image

Image(filename='gw_tutorial/BS.png') 

In order, to generate a nicer plot, we can explicitly set the energy window to an interesting region using the `-e` option and rescale the plot using the `-s` option. 

In [ ]:
%%bash
cd gw_tutorial
python -m excitingscripts.plot.band_structure -e -9 12 -s 1.5 1
mv PLOT.png BS.png
cd ..

In [ ]:
from IPython.display import Image
Image(filename='gw_tutorial/BS.png') 

<hr style="border:2px solid #DDD"> </hr>
This concludes our tutorial for the computation quasi-particle band structures with the GW approximation. You are encouraged to checkout the other methods implemented in exciting via the tutorials in this suite.
<hr style="border:2px solid #DDD"> </hr>